# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you in loading and exploring the FAIR² Mass Spectrometry dataset using the `mlcroissant` library—a Python toolkit for FAIR, explainable, and AI-ready data.

### Dataset Source
The dataset is accessed via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load the dataset's metadata and records with `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset and fetch the metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n\nPublished: {getattr(metadata, 'datePublished', 'n/a')}")

## 2. Data Overview
Explore available record sets, fields, and their `@id` identifiers.

We will list all record sets, their `@id`s, and available field `@id`s. These IDs will be used for targeted data extraction.

In [ ]:
# List all record sets in the dataset
print("Available Record Sets and their Field @id's:")
record_sets = list(metadata.record_sets)
for rs in record_sets:
    print(f"- Record Set name: {rs.name}")
    print(f"  Record Set @id: {rs.id}")
    print("  Fields:")
    for field in getattr(rs, 'fields', []):
        print(f"    - {field.name} (@id: {field.id}) | dataType: {getattr(field, 'data_type', 'n/a')}")
    print()

## 3. Data Extraction
Now we extract data from a record set for analysis. All entities are explicitly referenced by their `@id`.

> **Note:** The main data of interest containing protein abundance and properties appears in the primary record set, whose name is likely `ProteinAbundanceTable` (please confirm by inspecting the output above).

In [ ]:
# Collect all record set @id's
record_set_ids = [rs.id for rs in metadata.record_sets]

# Load each record set as a DataFrame
dataframes = {}
for record_set_id in record_set_ids:
    records_iter = dataset.records(record_set=record_set_id)
    records = list(records_iter)
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records from record set @id: {record_set_id}")
    else:
        print(f"No records found in record set @id: {record_set_id}")

# Print column names for primary data table (choose main @id, here we assume it's the first)
main_record_set_id = record_set_ids[0]  # <-- Replace with actual primary table @id if not first
print(f"\nColumns in DataFrame for {main_record_set_id}:")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply basic data manipulations—in this example, filtering on a numeric field (e.g., `mw_da` for molecular weight), normalizing, and grouping. All references are made by `@id`.

In [ ]:
# Pick a numeric field @id (e.g., "mw_da" for molecular weight in Dalton, verify with above 'columns')
numeric_field_id = None
for col in dataframes[main_record_set_id].columns:
    if 'mw' in col.lower() or 'mass' in col.lower():
        numeric_field_id = col
        break
if not numeric_field_id:
    # Fallback: Use the first numeric column found
    sample_row = dataframes[main_record_set_id].iloc[0]
    for col in dataframes[main_record_set_id].columns:
        if pd.api.types.is_numeric_dtype(dataframes[main_record_set_id][col]):
            numeric_field_id = col
            break

print(f"Using numeric field: {numeric_field_id}")
threshold = dataframes[main_record_set_id][numeric_field_id].mean()  # Use mean as threshold for demonstration

# Filter
filtered_df = dataframes[main_record_set_id][dataframes[main_record_set_id][numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
print(filtered_df.head(3))

# Normalize
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head(3))

# Pick a categorical/group field (commonly named 'description', 'accession', etc., check columns)
group_field_id = None
for col in dataframes[main_record_set_id].columns:
    if any(key in col.lower() for key in ['description', 'accession', 'category', 'group']):
        group_field_id = col
        break
if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
    print(grouped_df.head(3))

## 5. Visualization
Visualize the distribution of the numeric field and relationship with a selected category if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric field
plt.figure(figsize=(7,4))
sns.histplot(dataframes[main_record_set_id][numeric_field_id], bins=30, kde=True, color='teal')
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Frequency')
plt.show()

# If group field exists, boxplot of numeric field per group (show top 5 most frequent groups)
if group_field_id:
    top_groups = dataframes[main_record_set_id][group_field_id].value_counts().nlargest(5).index
    plt.figure(figsize=(10,5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=dataframes[main_record_set_id][dataframes[main_record_set_id][group_field_id].isin(top_groups)])
    plt.title(f"{numeric_field_id} by {group_field_id} (top 5 groups)")
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
This notebook demonstrated how to use the `mlcroissant` library to load, examine, process, and visualize the FAIR² mass spectrometry dataset. By referencing entities and data columns by their `@id` (as defined in the Croissant schema), the workflow remains robust and reproducible. Continue your analysis or modeling using these dataframes, and refer back to the schema for precise data semantics.